# Debug Scraper Surya Malang

Local HTML first, then optional live list/article checks. Output list CSV: `csv/surya_malang_list_debug.csv`.

In [1]:
from pathlib import Path
import sys
import importlib
from urllib.parse import urljoin

import pandas as pd
from bs4 import BeautifulSoup

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'scrapers').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scrapers.common import cutoff_date, parse_date, records_to_df
import scrapers.suryamalang as scraper
scraper = importlib.reload(scraper)

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 160)

SAMPLE_PATH = PROJECT_ROOT / 'html_samples/suryamalang.html'
OUTPUT_PATH = PROJECT_ROOT / 'csv/surya_malang_list_debug.csv'
MAX_PAGES = 200
TITLE_LIMIT = 90
cutoff = cutoff_date()

print('project:', PROJECT_ROOT)
print('module:', scraper.__file__)
print('sample:', SAMPLE_PATH)
print('cutoff:', cutoff.date())


project: /Users/tsar/Library/Mobile Documents/com~apple~CloudDocs/main/college/pkl/project
module: /Users/tsar/Library/Mobile Documents/com~apple~CloudDocs/main/college/pkl/project/scrapers/suryamalang.py
sample: /Users/tsar/Library/Mobile Documents/com~apple~CloudDocs/main/college/pkl/project/html_samples/suryamalang.html
cutoff: 2026-02-28


In [2]:

def short_title(value, limit=TITLE_LIMIT):
    text = '' if pd.isna(value) else str(value).strip()
    return text if len(text) <= limit else text[: limit - 3] + '...'


def ensure_debug_columns(df):
    df = df.copy()
    if 'page_num' not in df.columns:
        df['page_num'] = pd.NA
    if 'published_date' not in df.columns:
        df['published_date'] = None
    df['published_dt'] = df['published_date'].apply(parse_date)
    return df


def print_debug_rows(df):
    for _, row in df.iterrows():
        page = row.get('page_num')
        page_text = '---' if pd.isna(page) else f'{int(page):03d}'
        date_text = row.get('published_date')
        date_text = 'None' if pd.isna(date_text) else str(date_text)
        print(f"page={page_text} | date={date_text} | title={short_title(row.get('title'))}")


def audit_list(df):
    df = ensure_debug_columns(df)
    print('total rows:', len(df))
    if len(df) == 0:
        print('No rows found.')
        return df
    print('first page:', df['page_num'].dropna().min() if df['page_num'].notna().any() else None)
    print('last page:', df['page_num'].dropna().max() if df['page_num'].notna().any() else None)
    print('newest date:', df['published_dt'].max())
    print('oldest date:', df['published_dt'].min())
    print('cutoff date:', cutoff)
    print('reached cutoff:', bool((df['published_dt'].dropna() < cutoff).any()))
    print('null parsed dates:', int(df['published_dt'].isna().sum()))

    print('\nrows per month:')
    display(
        df.assign(month=df['published_dt'].dt.to_period('M').astype(str))
        .groupby('month', dropna=False)
        .size()
        .reset_index(name='count')
    )

    print('\nrows per page:')
    display(
        df.groupby('page_num', dropna=False)
        .agg(rows=('url', 'count'), newest=('published_dt', 'max'), oldest=('published_dt', 'min'))
        .reset_index()
        .tail(80)
    )
    return df


In [3]:

def parse_local_list(html_text):
    soup = BeautifulSoup(html_text, 'html.parser')
    rows = []
    for card in soup.select('ul.lsi > li'):
        title_tag = card.select_one('h3 a[href]')
        date_tag = card.select_one('div.grey span.grey, div.grey')
        excerpt_tag = card.select_one('h4.grey2')
        image_tag = card.select_one('img')
        if not title_tag:
            continue
        rows.append({
            'page_num': 1,
            'list_page_url': scraper.BASE_URL,
            'title': title_tag.get_text(' ', strip=True),
            'url': urljoin(scraper.BASE_URL, title_tag['href']),
            'published_date': scraper.clean_date(date_tag.get_text(' ', strip=True) if date_tag else None),
            'excerpt': excerpt_tag.get_text(' ', strip=True) if excerpt_tag else None,
            'image_url': image_tag.get('src') if image_tag else None,
        })
    return records_to_df(rows)


## Local HTML list parse

In [4]:
html_text = SAMPLE_PATH.read_text(errors='replace')
local_df = parse_local_list(html_text)
local_df = ensure_debug_columns(local_df)
print_debug_rows(local_df)
local_df = audit_list(local_df)


page=001 | date=5 hari lalu | title=Kronologi Kecelakaan Maut di Karangkates Kabupaten Malang, Tewaskan Pengendara Motor
page=001 | date=6 hari lalu | title=Akses Wisata Pantai Malang Selatan Semakin Mulus Jelang Libur Sekolah
page=001 | date=Jumat, 19 Juni 2026 | title=Transformasi Pergulaan Nasional Lewat Peremajaan Tebu Serentak di Jawa Timur
page=001 | date=Selasa, 26 Mei 2026 | title=6 Anggota Keluarga di Singosari Malang Jadi Korban Ledakan Dasyat, Diduga Racik Petasan
page=001 | date=Senin, 18 Mei 2026 | title=Dua Maling Curi Jeruk Petani 53 Kilogran di Turen Malang
page=001 | date=Jumat, 15 Mei 2026 | title=KRONOLOGI Pengendara Motor Tabrak Truk Tangki di Singosari Malang
page=001 | date=Senin, 11 Mei 2026 | title=PLN UIT JBM Perkuat Layanan Listrik Industri, Kunjungi Perusahaan di Malang
page=001 | date=Jumat, 8 Mei 2026 | title=Nasib Para Tersangka dan Korban Insiden Pengeroyokan di Malang, yang Positif Narkoba Di...
page=001 | date=Jumat, 8 Mei 2026 | title=Daftar 4 Tersangk

,month,count
0,2026-05,7
1,2026-06,3



rows per page:


,page_num,rows,newest,oldest
0,1,10,2026-06-24 12:21:51.097922,2026-05-07


## Live list scrape

In [5]:
live_df = scraper.scrape_list(cutoff, max_pages=MAX_PAGES)
live_df = ensure_debug_columns(live_df)
print_debug_rows(live_df)
live_df = audit_list(live_df)


Scraping Surya Malang page 1: https://surabaya.tribunnews.com/tag/kabupaten-malang
Scraping Surya Malang page 2: https://surabaya.tribunnews.com/tag/kabupaten-malang?page=2
Scraping Surya Malang page 3: https://surabaya.tribunnews.com/tag/kabupaten-malang?page=3
page=001 | date=5 hari lalu | title=Kronologi Kecelakaan Maut di Karangkates Kabupaten Malang, Tewaskan Pengendara Motor
page=001 | date=6 hari lalu | title=Akses Wisata Pantai Malang Selatan Semakin Mulus Jelang Libur Sekolah
page=001 | date=Jumat, 19 Juni 2026 | title=Transformasi Pergulaan Nasional Lewat Peremajaan Tebu Serentak di Jawa Timur
page=001 | date=Selasa, 26 Mei 2026 | title=6 Anggota Keluarga di Singosari Malang Jadi Korban Ledakan Dasyat, Diduga Racik Petasan
page=001 | date=Senin, 18 Mei 2026 | title=Dua Maling Curi Jeruk Petani 53 Kilogran di Turen Malang
page=001 | date=Jumat, 15 Mei 2026 | title=KRONOLOGI Pengendara Motor Tabrak Truk Tangki di Singosari Malang
page=001 | date=Senin, 11 Mei 2026 | title=PLN U

,month,count
0,2026-03,6
1,2026-04,3
2,2026-05,7
3,2026-06,3



rows per page:


,page_num,rows,newest,oldest
0,1,10,2026-06-24 12:22:07.342021,2026-05-07
1,2,9,2026-04-20 00:00:00.000000,2026-03-04


## Save debug list CSV

In [6]:
df_to_save = live_df if 'live_df' in globals() and len(live_df) else local_df
OUTPUT_PATH.parent.mkdir(exist_ok=True)
df_to_save.to_csv(OUTPUT_PATH, index=False)
print('saved:', OUTPUT_PATH)


saved: /Users/tsar/Library/Mobile Documents/com~apple~CloudDocs/main/college/pkl/project/csv/surya_malang_list_debug.csv


## Optional article sample check

In [7]:
sample_rows = (live_df if 'live_df' in globals() and len(live_df) else local_df).head(5)
article_rows = []
for index, row in sample_rows.iterrows():
    try:
        article = scraper.extract_article(row)
        article_rows.append(article)
        print(f"[{len(article_rows)}] content_len={len(article.get('content') or '')} | {short_title(article.get('title'))}")
    except Exception as error:
        print(f"failed sample article {index + 1}: {row.get('url')} | {error}")
article_df = pd.DataFrame(article_rows)
display(article_df[['title', 'published_date', 'content', 'url']].head() if len(article_df) else article_df)


[1] content_len=2721 | Kronologi Kecelakaan Maut di Karangkates Kabupaten Malang, Tewaskan Pengendara Motor
[2] content_len=2434 | Akses Wisata Pantai Malang Selatan Semakin Mulus Jelang Libur Sekolah
[3] content_len=2833 | Transformasi Pergulaan Nasional Lewat Peremajaan Tebu Serentak di Jawa Timur
[4] content_len=2006 | 6 Anggota Keluarga di Singosari Malang Jadi Korban Ledakan Dasyat, Diduga Racik Petasan
[5] content_len=2149 | Dua Maling Curi Jeruk Petani 53 Kilogran di Turen Malang


,title,published_date,content,url
0,"Kronologi Kecelakaan Maut di Karangkates Kabupaten Malang, Tewaskan Pengendara Motor","Tayang: Selasa, 23 Juni 2026 15:59 WIB",Ringkasan Berita:\nKecelakaan maut terjadi di Jalan Raya Nasional III Karangkates yang menewaskan seorang pengendara motor.\nKorban bernama Suwito meninggal...,https://surabaya.tribunnews.com/jawa-timur/1942701/kronologi-kecelakaan-maut-di-karangkates-kabupaten-malang-tewaskan-pengendara-motor
1,Akses Wisata Pantai Malang Selatan Semakin Mulus Jelang Libur Sekolah,"Tayang: Senin, 22 Juni 2026 15:08 WIB",Ringkasan Berita:\nArus wisata ke Pantai\nMalang Selatan\nmeningkat signifikan selama libur sekolah 2025/2026\nJalur utama Gondanglegi-Bantur sepanjang 31 k...,https://surabaya.tribunnews.com/travel/1942598/akses-wisata-pantai-malang-selatan-semakin-mulus-jelang-libur-sekolah
2,Transformasi Pergulaan Nasional Lewat Peremajaan Tebu Serentak di Jawa Timur,"Tayang: Jumat, 19 Juni 2026 07:59 WIB",Ringkasan Berita:\nGubernur Jawa Timur\nKhofifah Indar Parawansa\nmeluncurkan program\nbongkar ratoon\ntebu\nserentak di Malang\nProgram ini bertujuan mempe...,https://surabaya.tribunnews.com/jawa-timur/1942351/transformasi-pergulaan-nasional-lewat-peremajaan-tebu-serentak-di-jawa-timur
3,"6 Anggota Keluarga di Singosari Malang Jadi Korban Ledakan Dasyat, Diduga Racik Petasan","Tayang: Selasa, 26 Mei 2026 19:27 WIB","Ringkasan Berita:\nSebanyak 6 orang, yang tiga diantaranya anak menjadi korban luka dalam peristiwa ledakan dahsyat yang diduga berasal dari bahan petasan y...",https://surabaya.tribunnews.com/malang-raya/1940352/6-anggota-keluarga-di-singosari-malang-jadi-korban-ledakan-dasyat-diduga-racik-petasan
4,Dua Maling Curi Jeruk Petani 53 Kilogran di Turen Malang,"Tayang: Senin, 18 Mei 2026 14:20 WIB","Ringkasan Berita:\nPolisi tangkap JV (34) dan RW (18) usai mencuri\njeruk\ndi dua kebun warga\nTuren\n,\nKabupaten Malang\nMalang.\nBarang bukti 53 kg\ndiam...",https://surabaya.tribunnews.com/jawa-timur/1939626/dua-maling-curi-jeruk-petani-53-kilogran-di-turen-malang
